# 03. ML 모델 학습 및 비교

**목표:** Logistic Regression, Random Forest, XGBoost를 학습하고 성능을 비교합니다.

| 모델 | 특징 |
|------|------|
| Logistic Regression | 선형 기준 모델 (해석 용이) |
| Random Forest | 앙상블, 특성 중요도 분석 |
| XGBoost | 고성능 그래디언트 부스팅 |

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np

from teen_mind.data.preprocessor import MentalHealthPreprocessor
from teen_mind.models.ml_classifier import MLClassifier
from teen_mind.utils.visualization import (
    plot_model_comparison, plot_confusion_matrix, plot_feature_importance
)

print('라이브러리 로드 완료')

In [ ]:
# 저장된 전처리기 로드
preprocessor = MentalHealthPreprocessor.load('../models/saved/preprocessor.pkl')

# 처리된 데이터 로드
from teen_mind.data.loader import load_processed_data
import os

proc_files = [f for f in os.listdir('../data/processed') if f.endswith('.csv')]

# 또는 원본 데이터부터 다시 처리
from teen_mind.data.loader import load_raw_data
raw_files = [f for f in os.listdir('../data/raw') if f.endswith('.csv')]
df = load_raw_data(raw_files[0])
X_train, X_test, y_train, y_test = preprocessor.fit_transform(df)

print(f'학습: {X_train.shape}, 테스트: {X_test.shape}')

## 1. 모델 학습 (5-fold Stratified CV)

In [ ]:
clf = MLClassifier(random_state=42, n_jobs=-1)
cv_results = clf.fit(X_train, y_train, cv=5)
print('\n교차 검증 결과:')
print(cv_results)

## 2. 테스트 성능 평가

In [ ]:
results_df = clf.evaluate(X_test, y_test)
print('\n테스트 성능:')
print(results_df)

# 결과 저장
results_df.to_csv('../models/saved/model_comparison.csv', index=False)
print('\n결과 저장 완료')

In [ ]:
fig = plot_model_comparison(results_df)
fig.show()

## 3. 혼동 행렬

In [ ]:
import matplotlib.pyplot as plt

y_pred = clf.predict(X_test)
fig = plot_confusion_matrix(y_test, y_pred, class_names=['Low', 'Medium', 'High'])
plt.show()

## 4. 특성 중요도

In [ ]:
feature_names = [
    'age', 'social_media_hours', 'sleep_hours', 'physical_activity_hours',
    'depression_score', 'anxiety_score', 'stress_level', 'self_esteem_score',
    'online_support_access', 'family_support_score', 'gender', 'academic_performance'
]

importance_df = clf.get_feature_importance(feature_names)
importance_df.to_csv('../models/saved/feature_importance.csv', index=False)

fig = plot_feature_importance(importance_df, top_n=10)
fig.show()

## 5. 최고 모델 저장

In [ ]:
clf.save_best('../models/saved/best_ml_model.pkl')
clf.save_all('../models/saved/')
print(f'최고 모델: {clf.best_model_name}')
print('모든 모델 저장 완료!')